#  Student Dropout Prediction — XGBoost

**Dataset:** `student_dropout_dataset_v3.csv`  
**Task:** Binary Classification — Prediksi Dropout  
**Model:** XGBoost Classifier (eXtreme Gradient Boosting)

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
!pip install xgboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import xgboost as xgb
from xgboost import XGBClassifier

print('Libraries loaded ')
print('XGBoost version:', xgb.__version__)

---
## 1.  Cara Melihat Tipe Data

| Fungsi | Kegunaan |
|--------|----------|
| `df.info()` | Overview kolom, tipe data, jumlah non-null |
| `df.dtypes` | Tipe data per kolom |
| `df.describe()` | Statistik numerik (mean, std, min, max, quartile) |
| `df.isnull().sum()` | Jumlah missing values |
| `df.nunique()` | Jumlah nilai unik (berguna identifikasi nominal vs ordinal) |

**Tipe data di pandas:**
- `int64` / `float64` → Numerik (langsung pakai sebagai fitur)
- `object` → String/kategorik (perlu encoding)
- `bool` → Boolean (0/1)

In [ ]:
df = pd.read_csv('../student_dropout_dataset_v3.csv')
print('Shape:', df.shape)
print('\nTipe Data:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
# EDA: Distribusi target
print('Distribusi Dropout:')
print(df['Dropout'].value_counts())

# Korelasi dengan target
print('\nHeatmap Korelasi Numerik vs Dropout:')
num_cols = df.select_dtypes(include='number').columns.tolist()
corr = df[num_cols].corr()['Dropout'].drop('Dropout').sort_values(key=abs, ascending=False)
plt.figure(figsize=(8, 5))
corr.plot(kind='bar', color=['tomato' if x > 0 else 'steelblue' for x in corr])
plt.title('Korelasi Fitur Numerik dengan Dropout')
plt.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

---
## 2.  Dataset Bisa Digunakan Untuk Apa

| Tujuan | Target | Jenis |
|--------|--------|-------|
| **Prediksi Dropout** ← (ini) | `Dropout` | Binary Classification |
| Prediksi CGPA | `CGPA` | Regression |
| Prediksi Semester_GPA | `Semester_GPA` | Regression |
| Identifikasi faktor dominan dropout | - | Feature Importance / SHAP |
| Early Warning System | Prob Dropout | Threshold Classification |

---
## 3.  Kenapa XGBoost?

XGBoost adalah **Gradient Boosting** yang sangat efisien:
- Pohon dibangun **sekuensial** — setiap pohon belajar dari kesalahan pohon sebelumnya
- Regularisasi built-in (`lambda`, `alpha`) → mencegah overfitting
- Sangat cepat berkat optimasi hardware (parallel tree boosting)
- **State-of-the-art** untuk tabular data di kompetisi
- Built-in handling untuk missing values!

```
Random Forest: Banyak pohon belajar PARALEL, hasil di-voting
XGBoost: Banyak pohon belajar SEKUENSIAL, setiap pohon perbaiki error sebelumnya
```

In [ ]:
# ============================================================
# PREPROCESSING
# ============================================================
target_col = 'Dropout'
drop_cols = ['Student_ID']

df_proc = df.drop(columns=drop_cols).copy()
le = LabelEncoder()
for col in df_proc.select_dtypes(include='object').columns:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

X = df_proc.drop(columns=[target_col])
y = df_proc[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Hitung scale_pos_weight untuk imbalanced data
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight (untuk imbalanced): {scale_pos_weight:.2f}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
## 4.  Parameter Yang Bisa Diubah / Disetel

### Parameter Utama XGBClassifier:

| Parameter | Default | Penjelasan |
|-----------|---------|------------|
| `n_estimators` | 100 | Jumlah pohon (boosting rounds). Lebih banyak = lebih akurat tapi lambat |
| `learning_rate` / `eta` | 0.3 | Kecepatan belajar. Kecil = lebih akurat tapi butuh lebih banyak pohon |
| `max_depth` | 6 | Kedalaman max pohon. Lebih dalam = lebih kompleks, rentan overfit |
| `subsample` | 1.0 | Proporsi sampel untuk tiap pohon. < 1.0 = regularisasi |
| `colsample_bytree` | 1.0 | Proporsi fitur tiap pohon. < 1.0 = regularisasi |
| `min_child_weight` | 1 | Min jumlah sampel di leaf. Lebih besar = lebih conservative |
| `gamma` | 0 | Min loss reduction untuk split. Lebih besar = lebih conservative |
| `reg_alpha` | 0 | L1 regularization (Lasso). Memperkecil koefisien |
| `reg_lambda` | 1 | L2 regularization (Ridge). Regularisasi utama |
| `scale_pos_weight` | 1 | Untuk imbalanced: ratio negatif/positif |
| `objective` | `'binary:logistic'` | Fungsi loss |
| `eval_metric` | - | Metrik evaluasi selama training |
| `early_stopping_rounds` | - | Hentikan training jika tidak ada perbaikan |

In [ ]:
# ============================================================
# BUILD XGBoost MODEL
# ============================================================
model = XGBClassifier(
    n_estimators=300,            # Jumlah boosting rounds
    learning_rate=0.05,          # Kecepatan belajar (coba: 0.01, 0.05, 0.1, 0.3)
    max_depth=6,                 # Kedalaman pohon (coba: 3, 4, 5, 6, 8)
    subsample=0.8,               # 80% sampel per tree
    colsample_bytree=0.8,        # 80% fitur per tree
    min_child_weight=5,          # Regularisasi
    gamma=1,                     # Min gain untuk split
    reg_alpha=0.1,               # L1 regularization
    reg_lambda=1.0,              # L2 regularization
    scale_pos_weight=scale_pos_weight,  # Handle imbalanced
    objective='binary:logistic',
    eval_metric='auc',
    early_stopping_rounds=30,   # Stop jika tidak ada perbaikan 30 rounds
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

# Training dengan early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print(f'Model XGBoost berhasil ditraining ')
print(f'Best iteration: {model.best_iteration}')

In [ ]:
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

---
## 5.  Evaluasi Yang Dipakai

In [ ]:
print('=' * 50)
print(' EVALUASI MODEL XGBoost')
print('=' * 50)
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision : {precision_score(y_test, y_pred):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_pred_proba):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Tidak Dropout', 'Dropout']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Dropout', 'Dropout'],
            yticklabels=['Tidak Dropout', 'Dropout'], ax=axes[0])
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC={auc:.3f}')
axes[1].plot([0,1],[0,1],'navy',linestyle='--')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Feature Importance
feat_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
feat_imp.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 10 Feature Importance')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

### Panduan:
| Metrik | Target Ideal | Cukup | Buruk |
|--------|-------------|-------|-------|
| Accuracy | > 85% | 75–85% | < 75% |
| F1-Score | > 0.80 | 0.65–0.80 | < 0.65 |
| ROC-AUC | > 0.85 | 0.70–0.85 | < 0.70 |

### XGBoost + Early Stopping:
- Jika `best_iteration` jauh lebih kecil dari `n_estimators`, berarti model konvergen dengan sedikit pohon
- Training loss turun terus-menerus sedangkan validation loss naik → **overfitting** → pakai early stopping

### Cara baca Learning Curve:
- Train loss < Val loss dengan gap besar → Overfitting
- Train loss ≈ Val loss (keduanya tinggi) → Underfitting

In [ ]:
# Cek training vs test
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, y_pred)
print(f'Train Accuracy : {train_acc:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')
if train_acc - test_acc > 0.05:
    print('  Ada tanda overfitting. Gunakan lebih banyak regularisasi.')
else:
    print('  Model terlihat sehat!')

# Cross-validation
cv = cross_val_score(XGBClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=6,
    scale_pos_weight=scale_pos_weight, random_state=42, verbosity=0
), X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print(f'\nCross-Val AUC: {cv.mean():.4f} ± {cv.std():.4f}')

---
## 7.  Cara Mengoptimasi Model

### Panduan Parameter XGBoost:

| Masalah | Solusi |
|---------|--------|
| **Overfitting** | Naikkan `min_child_weight`, `gamma`, `reg_lambda`. Turunkan `max_depth`, `learning_rate`. Gunakan `subsample` < 1.0, `colsample_bytree` < 1.0 |
| **Underfitting** | Naikkan `n_estimators`, turunkan `min_child_weight` dan `gamma`. Kurangi regularisasi |
| **Imbalanced** | Set `scale_pos_weight`, gunakan `eval_metric='aucpr'` |
| **Lambat** | Kurangi `n_estimators`, gunakan `tree_method='hist'` |

### Tips:
1. Mulai dengan `n_estimators` besar dan `early_stopping_rounds` aktif
2. Tune `max_depth` dan `min_child_weight` pertama
3. Lalu tune `subsample` dan `colsample_bytree`
4. Terakhir tune `learning_rate` (turunkan, naikkan `n_estimators`)

In [ ]:
# Hyperparameter Tuning
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.5, 1]
}

xgb_tuner = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=scale_pos_weight,
                  objective='binary:logistic',
                  random_state=42, verbosity=0, n_jobs=-1),
    param_dist,
    n_iter=15,
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

xgb_tuner.fit(X_train, y_train)
print(' Best Params:', xgb_tuner.best_params_)
print(f' Best AUC: {xgb_tuner.best_score_:.4f}')

In [ ]:
best_model = xgb_tuner.best_estimator_
y_b = best_model.predict(X_test)
y_bp = best_model.predict_proba(X_test)[:, 1]
print(f'Post-Tuning → Accuracy: {accuracy_score(y_test,y_b):.4f} | F1: {f1_score(y_test,y_b):.4f} | AUC: {roc_auc_score(y_test,y_bp):.4f}')

---
## 8.  Cara Menyimpan Model

In [ ]:
os.makedirs('saved_models', exist_ok=True)

# Simpan dengan joblib
joblib.dump(best_model, 'saved_models/xgboost_dropout.pkl')
joblib.dump(list(X.columns), 'saved_models/feature_columns_xgb.pkl')
print(' XGBoost model tersimpan')

# Alternatif: simpan native XGBoost format (.json atau .ubj)
best_model.save_model('saved_models/xgboost_dropout.json')
print(' Model juga tersimpan di format native XGBoost (.json)')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
# Cara 1: Load dengan joblib
loaded_model = joblib.load('saved_models/xgboost_dropout.pkl')

# Cara 2: Load dari format native XGBoost
# loaded_model2 = XGBClassifier()
# loaded_model2.load_model('saved_models/xgboost_dropout.json')

loaded_cols = joblib.load('saved_models/feature_columns_xgb.pkl')
print('Model dimuat ')

# Data baru
new_student = pd.DataFrame([{
    'Age': 20, 'Gender': 0, 'Family_Income': 18000,
    'Internet_Access': 1, 'Study_Hours_per_Day': 1.5,
    'Attendance_Rate': 58, 'Assignment_Delay_Days': 8,
    'Travel_Time_Minutes': 60, 'Part_Time_Job': 1,
    'Scholarship': 0, 'Stress_Index': 9,
    'GPA': 1.5, 'Semester_GPA': 1.3, 'CGPA': 1.4,
    'Semester': 3, 'Department': 1, 'Parental_Education': 0
}])[loaded_cols]  # Pastikan urutan kolom benar

pred = loaded_model.predict(new_student)[0]
prob = loaded_model.predict_proba(new_student)[0]

print(f'\nPrediksi: {" DROPOUT" if pred == 1 else " TIDAK DROPOUT"}')
print(f'Probabilitas Dropout: {prob[1]:.2%}')

# Interpretasi risiko
if prob[1] >= 0.7:
    print('  RISIKO TINGGI — Perlu intervensi segera!')
elif prob[1] >= 0.4:
    print(' RISIKO SEDANG — Perlu monitoring lebih ketat')
else:
    print(' RISIKO RENDAH — Mahasiswa dalam kondisi baik')

In [ ]:
# Batch prediksi untuk banyak data sekaligus
# Contoh: prediksi untuk seluruh test set
batch_pred = loaded_model.predict(X_test)
batch_prob = loaded_model.predict_proba(X_test)[:, 1]

result_df = X_test.copy()
result_df['Predicted_Dropout'] = batch_pred
result_df['Dropout_Probability'] = batch_prob.round(3)
result_df['Risk_Level'] = pd.cut(batch_prob,
                                  bins=[0, 0.3, 0.6, 1.0],
                                  labels=['Rendah', 'Sedang', 'Tinggi'])

print('Distribusi Tingkat Risiko:')
print(result_df['Risk_Level'].value_counts())
result_df[['Predicted_Dropout', 'Dropout_Probability', 'Risk_Level']].head(10)